In [ ]:
import os
from getpass import getpass

os.environ["GH_TOKEN"] = getpass("Paste GitHub token: ")
os.environ["HF_TOKEN"] = getpass("Paste Hugging Face token: ")

In [ ]:
%cd /content
!rm -rf vnlegal-rag-v2
!git clone https://${GH_TOKEN}@github.com/hyperformancelabs/vnlegal-rag-v2.git
%cd vnlegal-rag-v2
!pip install -e .

In [ ]:
!git pull

In [ ]:
!gdown 1vkAnfAHNPEhu-aKrhR17RbyY3TYqXLzm
!mkdir -p data/processed
!unzip -o data_processed.zip -d data/processed
!rm -rf data/processed/__MACOSX data_processed.zip
!ls data/processed/

In [ ]:
!pip install -U huggingface_hub
!hf auth login --token ${HF_TOKEN}

In [ ]:
# Set batch sizes for target GPU
# T4=16GB, L4=24GB, A100=40GB
!python scripts/update_batch_size.py --vram auto

In [ ]:
# Train all 4 bi-encoder models
# vietnamese-bi-encoder (pyvi), multilingual-e5-base, granite-embedding-97m, embeddinggemma-300m
# Hyperparams from ref project: batch=64, lr=4e-5, weight_decay=0.02, 10 epochs, MNR loss
!python scripts/train_bi.py --config configs/train/bi_encoder_vietnamese.yaml

In [ ]:
!cat results/train/bi-encoder-vietnamese-scores.json

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

MODEL_DIR = "models/BiEncoder/vietnamese-bi-encoder/best"
REPO_ID = "phatvucoder/vietnamese-bi-encoder"

# Uploads everything inside your local folder to the repository
api.upload_folder(
    folder_path=MODEL_DIR,
    repo_id=REPO_ID,
    repo_type="model"
)

In [ ]:
!python scripts/train_bi.py --config configs/train/bi_encoder_e5.yaml

In [ ]:
!python scripts/train_bi.py --config configs/train/bi_encoder_granite.yaml

In [ ]:
!python scripts/train_bi.py --config configs/train/bi_encoder_gemma.yaml

In [ ]:
!python scripts/train_bi.py --config configs/train/bi_encoder_vietnamese_mp.yaml

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

MODEL_DIR = "models/BiEncoder/vietnamese-bi-encoder-mp/best"
REPO_ID = "phatvucoder/vietnamese-bi-encoder-mp"

# Uploads everything inside your local folder to the repository
api.upload_folder(
    folder_path=MODEL_DIR,
    repo_id=REPO_ID,
    repo_type="model"
)

In [ ]:
!ls models/BiEncoder/*/best/

In [ ]:
!git status

In [ ]:
!git config user.email "vhphatdz@gmail.com"
!git config user.name "phatv1"
!git add results/
!git commit -m "train: bi-encoder fine-tuning all 4 models"
!git push